In [1]:
import os
import numpy as np
import pandas as pd
import gc
from itertools import product


In [2]:
# ── Cache parameters — must match hm_foaf_experiment_sampled.ipynb ───────────
TARGET_USERS           = 1000
MIN_TRAIN_INTERACTIONS = 20
SEED                   = 42
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/7/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_t25_v20_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)
print(f"Cache tag: {cache_tag}")


Cache tag: u1000_m20_t25_v20_s42


In [3]:
# ── Load sampled H&M dataset from cache ──────────────────────────────────────
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")
train_df.head()


Loaded from cache: u1000_m20_t25_v20_s42
Total Users: 1000
Total Items: 10354


,customer_id,product_code,bought
0,837,3526,1
1,837,4408,1
2,837,2546,1
3,837,4908,1
4,837,555,1


In [4]:
# ── Build per-user train / val / test dicts ──────────────────────────────────
# Treat first three columns as (user_id, item_id, rating) regardless of names.
def df_to_dict(df):
    d = {}
    cols = list(df.columns[:3])
    for uid, grp in df.groupby(cols[0]):
        d[int(uid)] = grp[cols].values  # shape (n, 3): user_id, item_id, rating
    return d

train_dict = df_to_dict(train_df)
val_dict   = df_to_dict(val_df)
test_dict  = df_to_dict(test_df)

# Only keep users present in all three splits
common_users = sorted(set(train_dict) & set(val_dict) & set(test_dict))
train_dict = {u: train_dict[u] for u in common_users}
val_dict   = {u: val_dict[u]   for u in common_users}
test_dict  = {u: test_dict[u]  for u in common_users}

print(f"Active users (train∩val∩test): {len(common_users)}")
print(f"Train interactions: {sum(len(v) for v in train_dict.values())}")
print(f"Val   interactions: {sum(len(v) for v in val_dict.values())}")
print(f"Test  interactions: {sum(len(v) for v in test_dict.values())}")


Active users (train∩val∩test): 1000
Train interactions: 63081
Val   interactions: 15010
Test  interactions: 25055


In [5]:
# ── Local Learning MF ────────────────────────────────────────────────────────
# Each user trains a local MF model on their own data only.
# No communication. The item factors are private to each user.
# This is the worst-case baseline (lower bound on performance).

class LocalMF:
    """Single-user local matrix factorisation (no sharing).

    Parameters
    ----------
    n_items   : total item catalogue size (for consistent indexing)
    latent_dim: embedding dimension
    lr        : SGD learning rate
    reg       : L2 regularisation coefficient (applied to all factors)
    """
    def __init__(self, n_items, latent_dim=10, lr=0.01, reg=0.01):
        self.n_items    = n_items
        self.latent_dim = latent_dim
        self.lr         = lr
        self.reg        = reg
        # User has a single row vector; item factors are local copies
        self.u = np.random.normal(scale=0.1, size=latent_dim)
        self.Q = np.random.normal(scale=0.1, size=(n_items, latent_dim))  # item factors
        self.b = np.zeros(n_items)                                         # item biases

    def predict(self, item_id):
        return float(np.dot(self.u, self.Q[item_id]) + self.b[item_id])

    def train_epoch(self, interactions):
        """One SGD pass over this user's interactions.
        interactions: array of shape (n, 3) — [user_id, item_id, rating]
        """
        np.random.shuffle(interactions)
        mse = 0.0
        for row in interactions:
            item_id = int(row[1])
            rating  = float(row[2])
            pred    = self.predict(item_id)
            err     = rating - pred
            mse    += err ** 2
            # Gradients
            grad_u = -err * self.Q[item_id] + self.reg * self.u
            grad_q = -err * self.u           + self.reg * self.Q[item_id]
            grad_b = -err                    + self.reg * self.b[item_id]
            # Updates
            self.u          -= self.lr * grad_u
            self.Q[item_id] -= self.lr * grad_q
            self.b[item_id] -= self.lr * grad_b
        return np.sqrt(mse / len(interactions))

    def compute_rmse(self, interactions):
        mse = 0.0
        for row in interactions:
            item_id = int(row[1])
            rating  = float(row[2])
            mse    += (rating - self.predict(item_id)) ** 2
        return np.sqrt(mse / len(interactions))


In [6]:
# ── Training loop for all users ──────────────────────────────────────────────
import time

def run_local_learning(train_dict, val_dict, n_items,
                        latent_dim=10, lr=0.01, reg=0.01,
                        epochs=50, patience=5):
    """Train one LocalMF per user; return mean val RMSE per epoch and final models."""
    users = sorted(train_dict.keys())
    models = {u: LocalMF(n_items, latent_dim=latent_dim, lr=lr, reg=reg)
              for u in users}

    epoch_val_rmses = []
    epoch_times     = []
    best_val        = float('inf')
    no_improve      = 0

    for epoch in range(epochs):
        t0 = time.time()
        for u in users:
            models[u].train_epoch(train_dict[u].copy())  # copy so shuffle is local

        # Mean val RMSE across users
        val_rmses = [models[u].compute_rmse(val_dict[u]) for u in users]
        mean_val  = float(np.mean(val_rmses))
        epoch_val_rmses.append(mean_val)
        epoch_times.append(time.time() - t0)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Val RMSE: {mean_val:.4f} | "
                  f"Time: {epoch_times[-1]:.1f}s")

        # Early stopping on val RMSE
        if mean_val < best_val:
            best_val   = mean_val
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    return models, epoch_val_rmses, epoch_times


In [7]:
# ── Grid search ───────────────────────────────────────────────────────────────
param_grid = {
    "latent_dim": [8, 10, 16],
    "lr":         [0.001, 0.005, 0.01],
    "reg":        [0.01, 0.05],
}

best_val_rmse = float("inf")
best_config   = None
gs_results    = []

for dim, lr, reg in product(param_grid["latent_dim"],
                             param_grid["lr"],
                             param_grid["reg"]):
    print(f"\nConfig: latent_dim={dim}, lr={lr}, reg={reg}")
    _, val_curve, _ = run_local_learning(
        train_dict, val_dict, n_items,
        latent_dim=dim, lr=lr, reg=reg,
        epochs=50, patience=5
    )
    final_val = val_curve[-1]
    gs_results.append((dim, lr, reg, final_val))
    print(f"  Final Val RMSE: {final_val:.4f}")
    if final_val < best_val_rmse:
        best_val_rmse = final_val
        best_config   = (dim, lr, reg)
    gc.collect()

print(f"\nBest config: latent_dim={best_config[0]}, lr={best_config[1]}, "
      f"reg={best_config[2]}, Val RMSE={best_val_rmse:.4f}")



Config: latent_dim=8, lr=0.001, reg=0.01
  Epoch  10 | Val RMSE: 1.6571 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6571 | Time: 0.4s
  Early stopping at epoch 28
  Final Val RMSE: 1.6571

Config: latent_dim=8, lr=0.001, reg=0.05
  Epoch  10 | Val RMSE: 1.6575 | Time: 0.9s
  Epoch  20 | Val RMSE: 1.6575 | Time: 0.4s
  Early stopping at epoch 25
  Final Val RMSE: 1.6575

Config: latent_dim=8, lr=0.005, reg=0.01
  Epoch  10 | Val RMSE: 1.6566 | Time: 0.5s
  Early stopping at epoch 12
  Final Val RMSE: 1.6566

Config: latent_dim=8, lr=0.005, reg=0.05
  Early stopping at epoch 8
  Final Val RMSE: 1.6569

Config: latent_dim=8, lr=0.01, reg=0.01
  Epoch  10 | Val RMSE: 1.6573 | Time: 0.4s
  Early stopping at epoch 10
  Final Val RMSE: 1.6573

Config: latent_dim=8, lr=0.01, reg=0.05
  Early stopping at epoch 7
  Final Val RMSE: 1.6577

Config: latent_dim=10, lr=0.001, reg=0.01
  Epoch  10 | Val RMSE: 1.6566 | Time: 0.4s
  Epoch  20 | Val RMSE: 1.6566 | Time: 0.4s
  Epoch  30 | Val RMSE: 1.6566 | 

In [8]:
# ── Retrain best config, evaluate on test set ────────────────────────────────
print("Retraining best config on full train set ...")
best_models, val_curve, epoch_times = run_local_learning(
    train_dict, val_dict, n_items,
    latent_dim=best_config[0],
    lr=best_config[1],
    reg=best_config[2],
    epochs=50, patience=5
)

# Per-user test RMSE, then mean
test_rmses  = [best_models[u].compute_rmse(test_dict[u]) for u in sorted(best_models)]
mean_test   = float(np.mean(test_rmses))
print(f"\nTest RMSE (mean over users): {mean_test:.4f}")


Retraining best config on full train set ...
  Early stopping at epoch 6

Test RMSE (mean over users): 1.6799


In [9]:
# ── Grid search results ──────────────────────────────────────────────────────
gs_df = pd.DataFrame(gs_results,
                     columns=["latent_dim", "lr", "reg", "val_rmse"])
print("Best config:")
print(gs_df.loc[gs_df["val_rmse"].idxmin()])
gs_df


Best config:
latent_dim    10.000000
lr             0.001000
reg            0.010000
val_rmse       1.656611
Name: 6, dtype: float64


,latent_dim,lr,reg,val_rmse
0,8,0.001,0.01,1.657070
1,8,0.001,0.05,1.657455
2,8,0.005,0.01,1.656644
3,8,0.005,0.05,1.656899
4,8,0.010,0.01,1.657300
5,8,0.010,0.05,1.657749
6,10,0.001,0.01,1.656611
7,10,0.001,0.05,1.657007
8,10,0.005,0.01,1.657223
9,10,0.005,0.05,1.657084


In [10]:
# ── Save per-epoch log ───────────────────────────────────────────────────────
log = {
    "epoch":    list(range(len(val_curve))),
    "val_rmse": val_curve,
    "time":     epoch_times,
}
log_df = pd.DataFrame(log)
log_df.to_csv("ll_hm.csv", index=False)
print(f"Saved ll_hm.csv  ({len(log_df)} epochs)")
log_df.head(10)


Saved ll_hm.csv  (6 epochs)


,epoch,val_rmse,time
0,0,1.656810,0.426297
1,1,1.656815,0.410153
2,2,1.656821,0.403018
3,3,1.656828,0.482578
4,4,1.656834,0.475652
5,5,1.656840,0.417164
